# 05: decision tree

**The algorithm.** Repeatedly split the data on the feature and threshold that best separate the classes, until a depth limit.

**Where the time goes.** For every node, try every feature and every cut point, and count classes on each side. There's no matrix math at all: it's branchy, integer counting work. GPUs and BLAS libraries can't help, which is exactly the case where **compiled code is supposed to win** (see the lesson on bottlenecks).

**Two different algorithms, on purpose.**
- scikit-learn's `DecisionTreeClassifier` sorts each feature's values and tries every exact cut.
- Our engines use the **histogram** method from LightGBM: bucket each feature into at most 256 bins once, then only count bins. It's much less work per node, and accuracy is usually about the same.

So this race mixes "better algorithm" with "faster engine". The `numpy` vs `rust` comparison isolates the engine: they run the identical algorithm and grow node-for-node identical trees (checked below).

In [ ]:
import sys
sys.path.insert(0, ".")  # so bench_utils (next to this notebook) imports

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ml_in_rust as mr
from bench_utils import (machine_info, timeit, load_covtype, add_speedup,
                         save_results, line_chart, fmt_ms, only_available)

pd.set_option("display.float_format", "{:,.4f}".format)
%matplotlib inline
pd.Series(machine_info())

In [ ]:
import os
QUICK = os.environ.get("QUICK") == "1"  # QUICK=1 ./run_all.sh for a fast smoke run
from sklearn.tree import DecisionTreeClassifier as SkTree

X, y = load_covtype()
if QUICK:
    X, y = X[:60_000], y[:60_000]
X_test, y_test = X[:50_000], y[:50_000]
X_train, y_train = X[50_000:], y[50_000:]

def race(depth, n):
    Xt, yt = X_train[:n], y_train[:n]
    models = only_available({
        "sklearn": lambda: SkTree(max_depth=depth, random_state=0),
        "numpy": lambda: mr.DecisionTreeClassifier(depth, backend="numpy"),
        "rust": lambda: mr.DecisionTreeClassifier(depth, backend="rust"),
    })
    out, trees = [], {}
    for name, make in models.items():
        fitted = {}
        def run():
            fitted["m"] = make().fit(Xt, yt)
        fit_s = timeit(run, repeats=1, warmup=False)
        m = fitted["m"]
        trees[name] = m
        out.append({"depth": depth, "train rows": n, "engine": name, "fit s": fit_s,
                    "predict s": timeit(lambda: m.predict(X_test), repeats=3),
                    "test accuracy": m.score(X_test, y_test),
                    "nodes": m.tree_.node_count if name == "sklearn" else m.node_count})
    a, b = trees["numpy"].tree_, trees["rust"].tree_
    identical = all(np.array_equal(a[key], b[key]) for key in a)
    return out, identical
print(f"train {len(X_train):,} rows, test {len(X_test):,} rows")

## Race 1: deeper trees (all training rows)

In [ ]:
rows, same = [], []
for depth in ([6, 10] if QUICK else [8, 12, 16, 20]):
    r, identical = race(depth, len(X_train))
    rows += r
    same.append({"depth": depth, "numpy and rust trees identical": identical})
by_depth = add_speedup(pd.DataFrame(rows), "fit s", ["depth"])
line_chart(by_depth, "depth", "fit s", "Decision tree: fit time as depth grows (531k rows)",
           "max depth", "seconds (log scale)", logx=False)
plt.show()
display(pd.DataFrame(same))
by_depth

## Race 2: more training rows (depth 16)

In [ ]:
rows = []
for n in ([20_000, 50_000] if QUICK else [50_000, 150_000, len(X_train)]):
    rows += race(16 if not QUICK else 10, n)[0]
by_rows = add_speedup(pd.DataFrame(rows), "fit s", ["train rows"])
line_chart(by_rows, "train rows", "fit s", "Decision tree: fit time as rows grow (depth 16)",
           "training rows", "seconds (log scale)")
plt.show()
by_rows.pivot(index="train rows", columns="engine", values="vs sklearn").round(2)

## Prediction speed

Walking a row down a tree is pointer-chasing: each step jumps to an unpredictable place in memory.

In [ ]:
pred = add_speedup(by_depth.rename(columns={"vs sklearn": "fit vs sklearn"}), "predict s", ["depth"])
pred.pivot(index="depth", columns="engine", values="vs sklearn").round(2)

In [ ]:
save_results("tree", by_depth, notes={"by_rows": by_rows.to_dict(orient="records")})
summary = by_depth[by_depth["depth"] == by_depth["depth"].max()].set_index("engine")
summary[["fit s", "vs sklearn", "test accuracy", "nodes"]]